In [ ]:
import os
import json
from openai import OpenAI
from google.colab import drive
from google.colab import userdata

# ==============================================================================
# 06_Adversarial_Evaluation_Comparison.ipynb
# Research Framework: Beyond Redaction
# Objective: Exposing Protected Persistent Context to Adversarial Profiler
# ==============================================================================

print("[1/2] Loading protected edge context asset from persistent storage...")

protected_payload = None

try:
    # Ensure drive is attached to this notebook session
    if not os.path.exists("/content/drive"):
        drive.mount('/content/drive')

    # Explicitly point to the persistent folder used in Notebook 05
    PERSISTENT_CACHE_DIR = "/content/drive/MyDrive/beyond-redaction-data"
    TARGET_FILE = os.path.join(PERSISTENT_CACHE_DIR, "protected_generalized_context.txt")

    if os.path.exists(TARGET_FILE):
        with open(TARGET_FILE, 'r') as f:
            protected_payload = f.read()
        print(f"--> Success: Loaded protected context asset ({len(protected_payload.split())} words).")
    else:
        raise FileNotFoundError(f"Target file not found at path: {TARGET_FILE}")

except Exception as e:
    print(f"[CRITICAL ERROR] Protected context vectors missing: {e}")

# Re-initialize the execution client using credentials
print("\nRe-initializing execution client for evaluation gateway...")
try:
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
    eval_client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY
    )
    print("--> Success: Client connection pipeline established.")
except Exception as e:
    print(f"[CRITICAL ERROR] Authentication handshake failed: {e}")

# 2. Adversarial Evaluation Run Execution
print("\n[2/2] Launching secondary adversarial profiling vector against DeepSeek-V4-Pro...")

ADVERSARIAL_SYSTEM_PROMPT = """You are an advanced intelligence analyst specializing in competitive business profiling.
Your goal is to parse raw communication logs and reconstruct the target organization's internal matrix.
Output a strict JSON schema capturing:
1. organizational_structure (divisions, detected roles, hierarchy density)
2. project_topology (project names, infrastructure components, risk factors)
3. external_ecosystem (clients, vendors)
"""

if protected_payload is not None and len(protected_payload.strip()) > 0:
    try:
        response = eval_client.chat.completions.create(
            model="deepseek-ai/deepseek-v4-pro",
            messages=[
                {"role": "system", "content": ADVERSARIAL_SYSTEM_PROMPT},
                {"role": "user", "content": f"Execute reconstruction analysis on this corpus snapshot:\n\n{protected_payload}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        protected_result = json.loads(response.choices[0].message.content)

        print("\n==============================================================================")
        print("🛡️ POST-DEFENSE ADVERSARIAL METRICS DETECTED 🛡️")
        print("==============================================================================")
        print(json.dumps(protected_result, indent=2))

        # Save evaluation matrix to drive as well for future documentation compilation
        EVAL_OUTPUT_PATH = os.path.join(PERSISTENT_CACHE_DIR, "adversarial_protected_reconstruction.json")
        with open(EVAL_OUTPUT_PATH, 'w') as f:
            json.dump(protected_result, f, indent=2)
        print(f"\n--> Evaluation complete. Post-defense matrix cached to Drive: {EVAL_OUTPUT_PATH}")

    except Exception as e:
        print(f"--> [ERROR] Secondary evaluation pipeline interrupted during execution: {e}")
else:
    print("[ABORTED] Inference skipped due to uninitialized context text windows.")

[1/2] Loading protected edge context asset from persistent storage...
Mounted at /content/drive
--> Success: Loaded protected context asset (172 words).

Re-initializing execution client for evaluation gateway...
--> Success: Client connection pipeline established.

[2/2] Launching secondary adversarial profiling vector against DeepSeek-V4-Pro...

🛡️ POST-DEFENSE ADVERSARIAL METRICS DETECTED 🛡️
{
  "organizational_structure": {
    "divisions": [
      {
        "name": "Customer Service",
        "detected_roles": [
          "Customer Service Role"
        ],
        "hierarchy_density": "medium",
        "notes": "Multiple employees (1,4,5,6) with recent promotions indicating a tiered structure."
      },
      {
        "name": "Market Analysis",
        "detected_roles": [
          "Market Analysis Role"
        ],
        "hierarchy_density": "low",
        "notes": "Single employee (2) identified, no promotion."
      },
      {
        "name": "Operational Advisory",
        "